In [ ]:
import sys
import subprocess

def ensure_package(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('datasets')
ensure_package('pandas')
ensure_package('numpy')
ensure_package('scikit-learn', 'sklearn')

In [ ]:
import random
import time
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 200)

In [ ]:
dataset = load_dataset('emotion')
print(dataset)
print('Splits:', list(dataset.keys()))
for split in dataset.keys():
    print(split, len(dataset[split]))

In [ ]:
label_feature = dataset['train'].features['label']
label_names = label_feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}
print('Labels:', id2label)

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

for df in [train_df, val_df, test_df]:
    df['label_name'] = df['label'].map(id2label)
    df['text_clean'] = df['text'].astype(str).str.strip().str.lower()
    df['char_len'] = df['text'].astype(str).str.len()
    df['word_len'] = df['text'].astype(str).str.split().str.len()

print(train_df.head(10))

In [ ]:
print('Train label distribution:')
print(train_df['label_name'].value_counts().sort_index())
print('\nValidation label distribution:')
print(val_df['label_name'].value_counts().sort_index())
print('\nTest label distribution:')
print(test_df['label_name'].value_counts().sort_index())

print('\nText length stats (train):')
print(train_df[['char_len', 'word_len']].describe())

In [ ]:
X_train = train_df['text_clean'].tolist()
y_train = train_df['label'].tolist()
X_val = val_df['text_clean'].tolist()
y_val = val_df['label'].tolist()
X_test = test_df['text_clean'].tolist()
y_test = test_df['label'].tolist()

models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ('clf', LinearSVC(random_state=SEED))
    ]),
    'Multinomial NB': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ('clf', MultinomialNB())
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.95, sublinear_tf=True, max_features=5000)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
    ])
}

print('Models to evaluate:', list(models.keys()))

In [ ]:
results = []
fitted_models = {}
predictions = {}

for model_name, model in models.items():
    print('=' * 100)
    print('Training model:', model_name)
    
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    start_val = time.time()
    val_preds = model.predict(X_val)
    val_infer_time = time.time() - start_val
    
    start_test = time.time()
    test_preds = model.predict(X_test)
    test_infer_time = time.time() - start_test
    
    val_acc = accuracy_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds, average='macro')
    test_acc = accuracy_score(y_test, test_preds)
    test_f1 = f1_score(y_test, test_preds, average='macro')
    
    results.append({
        'model': model_name,
        'val_accuracy': val_acc,
        'val_macro_f1': val_f1,
        'test_accuracy': test_acc,
        'test_macro_f1': test_f1,
        'train_time_sec': train_time,
        'val_inference_sec': val_infer_time,
        'test_inference_sec': test_infer_time
    })
    
    fitted_models[model_name] = model
    predictions[model_name] = {
        'val_preds': val_preds,
        'test_preds': test_preds
    }
    
    print(f'Training time: {train_time:.3f}s')
    print(f'Validation inference time: {val_infer_time:.3f}s')
    print(f'Test inference time: {test_infer_time:.3f}s')
    print(f'Validation Accuracy: {val_acc:.4f}')
    print(f'Validation Macro F1: {val_f1:.4f}')
    print(f'Test Accuracy: {test_acc:.4f}')
    print(f'Test Macro F1: {test_f1:.4f}')

In [ ]:
results_df = pd.DataFrame(results).sort_values(['test_macro_f1', 'test_accuracy'], ascending=False).reset_index(drop=True)
print(results_df)

best_model_name = results_df.iloc[0]['model']
best_model = fitted_models[best_model_name]
best_val_preds = predictions[best_model_name]['val_preds']
best_test_preds = predictions[best_model_name]['test_preds']

print('\nBest model:', best_model_name)

In [ ]:
print('Validation classification report for best model:')
print(classification_report(y_val, best_val_preds, target_names=label_names, digits=4))

print('Test classification report for best model:')
print(classification_report(y_test, best_test_preds, target_names=label_names, digits=4))

cm = confusion_matrix(y_test, best_test_preds)
cm_df = pd.DataFrame(cm, index=[f'true_{x}' for x in label_names], columns=[f'pred_{x}' for x in label_names])
print('Test confusion matrix for best model:')
print(cm_df)

In [ ]:
error_df = test_df[['text', 'text_clean', 'label', 'label_name', 'char_len', 'word_len']].copy()
error_df['pred'] = best_test_preds
error_df['pred_name'] = error_df['pred'].map(id2label)
error_df['correct'] = error_df['label'] == error_df['pred']

misclassified = error_df[~error_df['correct']].copy()
print('Best model:', best_model_name)
print('Total test examples:', len(error_df))
print('Misclassified test examples:', len(misclassified))

print('Top confusion pairs:')
print(misclassified.groupby(['label_name', 'pred_name']).size().sort_values(ascending=False).head(20))

print('\nSample misclassifications:')
print(misclassified[['text', 'label_name', 'pred_name', 'word_len']].head(25).to_string(index=False))

In [ ]:
def predict_emotion(texts, model_name=None):
    chosen_name = model_name or best_model_name
    chosen_model = fitted_models[chosen_name]
    cleaned = [str(t).strip().lower() for t in texts]
    pred_ids = chosen_model.predict(cleaned)
    
    confidence = [None] * len(cleaned)
    if hasattr(chosen_model, 'predict_proba'):
        try:
            probas = chosen_model.predict_proba(cleaned)
            confidence = probas.max(axis=1)
        except Exception:
            confidence = [None] * len(cleaned)
    
    return pd.DataFrame({
        'model': [chosen_name] * len(texts),
        'text': texts,
        'pred_label_id': pred_ids,
        'pred_label': [id2label[i] for i in pred_ids],
        'confidence': confidence
    })

sample_texts = [
    'i feel amazing and grateful today',
    'i am really upset and angry about what happened',
    'i miss my friends and feel lonely',
    'i am scared about tomorrow',
    'this was such a lovely surprise'
]

print(predict_emotion(sample_texts).to_string(index=False))